# Kandinsky 6 T2VA → SR — stacked Diffusers example

This notebook follows the two-stage pattern used by LTX-2.5: load the T2VA and SR artifacts independently, generate video and audio with T2VA, then pass the decoded video and generated audio to SR. The pipelines remain separate and can be loaded, offloaded, or replaced independently.

The SR artifact is created with `convert_sr_checkpoint.py` and is a separately packaged Diffusers bundle containing `Kandinsky6SRPipeline`. The production SR route is `resolution_scale=2.25` (1.125x pixel pre-upscale followed by the x2 KVAE path).

In [ ]:
from pathlib import Path

import torch
from diffusers import Kandinsky6SRPipeline, Kandinsky6TI2VAPipeline
from diffusers.utils import encode_video

REPO = Path.cwd()
while REPO != REPO.parent and not (REPO / "dev" / "exports" / "diffusers").exists():
    REPO = REPO.parent
if not (REPO / "dev" / "exports" / "diffusers").exists():
    raise RuntimeError("Run this notebook from the K6 repository or one of its subdirectories.")

In [ ]:
# Stage 1 and stage 2 are independent Diffusers artifacts.
T2VA_MODEL_PATH = REPO / "outputs" / "diffusers_bundle" / "kandinsky6_diffusers"
SR_MODEL_PATH = REPO / "outputs" / "diffusers_bundle" / "kandinsky6_sr_diffusers"
GPU_ID = 0  # Change this to the CUDA device used for inference.

t2va_pipe = Kandinsky6TI2VAPipeline.from_pretrained(
    T2VA_MODEL_PATH,
    torch_dtype=torch.bfloat16,
)
t2va_pipe.enable_model_cpu_offload(gpu_id=GPU_ID)

sr_pipe = Kandinsky6SRPipeline.from_pretrained(
    SR_MODEL_PATH,
    torch_dtype=torch.bfloat16,
)
sr_pipe.enable_model_cpu_offload(gpu_id=GPU_ID)

print(f"T2VA artifact: {T2VA_MODEL_PATH}")
print(f"SR artifact: {SR_MODEL_PATH}")

In [ ]:
prompt = "A cinematic shot of a small red sailboat crossing a calm blue lake at sunset, with gentle waves and natural ambient sound."
negative_prompt = "blurry, distorted, low quality, noisy audio"
sample_fps = 24.0

t2va_result = t2va_pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    height=480,
    width=864,
    num_frames=121,  # 1 + 8k: Kandinsky6SRPipeline rejects other values
    sample_fps=sample_fps,
    num_inference_steps=50,
    guidance_scale=5.0,
    sample_audio=True,
    output_type="pt",
)
if t2va_result.audio is None:
    raise RuntimeError("The T2VA pipeline returned no audio")
print(f"T2VA frames: {tuple(t2va_result.frames.shape)}")
print(f"T2VA audio streams: {len(t2va_result.audio)}")

In [ ]:
# Stage 2 consumes the stage-1 tensors directly; no intermediate video encode/decode is needed.
output_path = REPO / "outputs" / "diffusers" / "kandinsky6_t2va_sr.mp4"
output_path.parent.mkdir(parents=True, exist_ok=True)

sr_result = sr_pipe(
    video=t2va_result.frames,
    resolution_scale=2.25,
    output_type="pt",
)
print(f"SR frames: {tuple(sr_result.frames.shape)}")

audio = torch.as_tensor(t2va_result.audio[0])[:, None].repeat(1, 2)
encode_video(
    sr_result.frames[0].permute(1, 2, 3, 0),
    fps=int(sample_fps),
    output_path=str(output_path),
    audio=audio,
    audio_sample_rate=int(t2va_pipe.audio_sample_rate),
)
print(f"saved: {output_path}")